In [2]:
# ==============================================================================
# NEXUS OPTIML 2026: UNIFIED OPTIMAL PRODUCT RETURN PREDICTION PIPELINE
# INTEGRATING:
#   1. Physics & GBDT Multi-Table Imputation (Zero Information Loss)
#   2. Cents-vs-Dollars Anomaly Normalization
#   3. Multi-Item Order Bracketing Detectors
#   4. Promo Channel Code Temporal Drift Neutralization (Logit Offset)
#   5. Purged Rolling Time-Based Cross-Validation (Grouped by Order ID)
#   6. Adaptive Out-Of-Fold Bayesian Target Encodings
#   7. Automated Hyperparameter Tuning (Optuna / Pre-tuned GBDT + RF + SGD)
#   8. Quintuple Model Diversity Ensemble (LGBM + CatBoost + XGBoost + RF + SGD)
#   9. Calibration-Invariant Percentile Rank-Average Blending
# ==============================================================================

import gc
import os
import re
import time
import glob
import difflib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

# Gradient Boosted Decision Tree Frameworks
import lightgbm as lgb

try:
    from catboost import CatBoostClassifier, Pool
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 120)

# ------------------------------------------------------------------------------
# 0. CONFIGURATION & ENVIRONMENT SETUP
# ------------------------------------------------------------------------------
SEED = 42
N_JOBS = -1
cands = [os.environ.get('DATA_DIR', '.'), '.', '/kaggle/input/*/', '/kaggle/input/*/*/']
DATA_DIR = next((d for d in cands if os.path.exists(os.path.join(d, 'test.csv'))), '.')

print(f"Data Directory    : {DATA_DIR}")
print(f"LightGBM Version  : {lgb.__version__}")
print(f"CatBoost Ready    : {CATBOOST_AVAILABLE}")
print(f"XGBoost Ready     : {XGBOOST_AVAILABLE}")
print(f"Optuna Ready      : {OPTUNA_AVAILABLE}")


# ==============================================================================
# 1. MULTI-TABLE LOGICAL IMPUTATION & CANONICALIZATION
# ==============================================================================
print("\n--- [Step 1/8] Cleaning & Logically Imputing Supplementary Tables ---")

re_clean = lambda s: re.sub(r'[^a-z]', '', str(s).lower())

def canon_map(values, canon, cutoff=0.75):
    """Maps dirty spelling variants and typos to canonical categories/regions."""
    ck = {re_clean(c): c for c in canon}
    out = {}
    for v in pd.unique(values):
        k = re_clean(v)
        if k in ck:
            out[v] = ck[k]
        else:
            matches = difflib.get_close_matches(k, list(ck), 1, cutoff)
            out[v] = ck[matches[0]] if matches else v
    return out

# --- 1.1 Customers: Region standardization, deduplication & tenure history ---
df_cust = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
df_stores = pd.read_csv(os.path.join(DATA_DIR, 'stores.csv'))

st_regions = list(df_stores['region'].unique())
df_stores['region'] = df_stores['region'].map(canon_map(df_stores['region'].unique(), st_regions))
df_cust['region'] = df_cust['region'].map(canon_map(df_cust['region'].unique(), st_regions))

df_cust['record_updated_at'] = pd.to_datetime(df_cust['record_updated_at'], errors='coerce')
df_cust = df_cust.sort_values('record_updated_at')
df_cust['n_records'] = df_cust.groupby('customer_id')['customer_id'].transform('size').astype('int8')

# Retain first loyalty tier to detect loyalty status elevation over time
first_tier = df_cust.groupby('customer_id')['loyalty_tier'].first().rename('tier_first')
df_cust = df_cust.groupby('customer_id', as_index=False).last().merge(first_tier, on='customer_id')
df_cust['tier_changed'] = (df_cust['tier_first'] != df_cust['loyalty_tier']).astype('int8')
df_cust['age_missing'] = df_cust['age_band'].isnull().astype('int8')

# Logical mode imputation for age_band from (customer_type x loyalty_tier)
age_mode = df_cust.groupby(['customer_type', 'loyalty_tier'])['age_band'].transform(
    lambda s: s.mode().iat[0] if s.notnull().any() else np.nan
)
df_cust['age_band'] = df_cust['age_band'].fillna(age_mode).fillna('unknown')
df_cust['preferred_contact'] = df_cust['preferred_contact'].fillna('none')
df_cust['newsletter_opt_in'] = df_cust['newsletter_opt_in'].fillna(0).astype('int8')
df_cust['signup_date'] = pd.to_datetime(df_cust['signup_date'], errors='coerce')

# --- 1.2 Products: Category canonicalization (all 12 classes) & density recovery ---
df_prod = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
clean_cat = df_prod['category'].astype(str).str.lower().str.replace(r'[^a-z]', '', regex=True)
vc_cat = clean_cat.value_counts()
# Preserves all 12 genuine retail classes (including furniture & food_beverages)
canonical_cats = vc_cat[vc_cat >= 300].index.tolist()
cat_map = {c: (c if c in canonical_cats else (difflib.get_close_matches(c, canonical_cats, 1, 0.5) or [c])[0]) for c in vc_cat.index}
df_prod['category_clean'] = clean_cat.map(cat_map)
df_prod['vol_cm3'] = df_prod['length_cm'] * df_prod['width_cm'] * df_prod['height_cm']

# Physical density (kg / cm^3) recovery
known_w = df_prod.dropna(subset=['weight_kg']).copy()
known_w['density'] = known_w['weight_kg'] / (known_w['vol_cm3'] + 1e-5)
cat_dens = known_w.groupby('category_clean')['density'].median()
over_dens = known_w['density'].median()

missing_w = df_prod['weight_kg'].isnull()
df_prod.loc[missing_w, 'weight_kg'] = (
    df_prod.loc[missing_weight, 'vol_cm3'] * df_prod.loc[missing_weight, 'category_clean'].map(cat_dens).fillna(over_dens)
)
df_prod['weight_is_imputed'] = missing_w.astype('int8')
df_prod['density'] = (df_prod['weight_kg'] / (df_prod['vol_cm3'] + 1e-4)).astype('float32')

# Relative category benchmarks
cat_benchmarks = df_prod.groupby('category_clean').agg(
    cat_median_price=('list_price', 'median'),
    cat_median_weight=('weight_kg', 'median')
).reset_index()
df_prod = df_prod.merge(cat_benchmarks, on='category_clean', how='left')
df_prod['price_to_cat_median'] = (df_prod['list_price'] / (df_prod['cat_median_price'] + 1e-4)).astype('float32')
df_prod['weight_to_cat_median'] = (df_prod['weight_kg'] / (df_prod['cat_median_weight'] + 1e-4)).astype('float32')
df_prod.drop(columns=['category'], inplace=True)

# --- 1.3 Stores: Format-driven floor space imputation ---
df_stores['is_physical_store'] = df_stores['channel'].isin(['store', 'catalog']).astype('int8')
df_stores['size_missing'] = df_stores['size_sqm'].isnull().astype('int8')
df_stores['size_sqm'] = df_stores['size_sqm'].fillna(
    df_stores.groupby('format')['size_sqm'].transform('median')
).fillna(0.0)
df_stores['opened_date'] = pd.to_datetime(df_stores['opened_date'], errors='coerce')

# --- 1.4 Calendar: Stationary economic changes & weather deviations ---
df_cal = pd.read_csv(os.path.join(DATA_DIR, 'calendar.csv'))
df_cal['week_start'] = pd.to_datetime(df_cal['week_start'])
df_cal = df_cal.sort_values('week_start')
for c in ['cpi', 'fuel_price', 'unemployment']:
    df_cal[f'{c}_chg'] = df_cal[c].diff().fillna(0).astype('float32')
df_cal['temp_dev'] = (df_cal['temperature_f'] - df_cal['temperature_f'].rolling(52, min_periods=1, center=True).mean()).fillna(0).astype('float32')

print(f"Products missing weight after density recovery : {df_prod['weight_kg'].isnull().sum()}")
print(f"Stores missing size_sqm after format recovery    : {df_stores['size_sqm'].isnull().sum()}")
print(f"Customers missing age_band after mode tagging   : {df_cust['age_band'].isnull().sum()}")


# ==============================================================================
# 2. FEATURE ENGINEERING (BRACKETING, ANOMALIES & ECONOMIC RATIOS)
# ==============================================================================
print("\n--- [Step 2/8] Engineering Bracketing, Pricing & Behavioral Features ---")

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df['order_ts'] = pd.to_datetime(df['order_ts'])
    df['week_start'] = df['order_ts'].dt.floor('D') - pd.to_timedelta(df['order_ts'].dt.dayofweek, unit='D')

    # Relational joins
    df = df.merge(df_cust.drop(columns=['record_updated_at']).rename(columns={'region': 'c_region', 'customer_type': 'c_type'}), on='customer_id', how='left')
    df = df.merge(df_prod.rename(columns={'category_clean': 'cat'}), on='product_id', how='left')
    df = df.merge(df_stores.rename(columns={'channel': 's_channel', 'region': 's_region', 'format': 's_format'}), on='store_id', how='left')
    df = df.merge(df_cal[['week_start', 'holiday_flag', 'temperature_f', 'fuel_surcharge', 'cpi_chg', 'fuel_price_chg', 'unemployment_chg', 'temp_dev', 'cpi', 'fuel_price']], on='week_start', how='left')

    # 1. Price Anomaly Correction: Cents-vs-Dollars Bug
    ratio = df['unit_price'] / (df['list_price'] + 1e-4)
    df['is_price_in_cents'] = ((ratio >= 60.0) & (ratio <= 140.0)).astype('int8')
    df['unit_price_clean'] = np.where(df['is_price_in_cents'] == 1, df['unit_price'] / 100.0, df['unit_price']).astype(np.float32)

    # 2. Logical Transaction Attribute Imputations
    for c, f in [('discount', 'discount_missing'), ('freight_cost', 'freight_missing'), ('session_clicks', 'clicks_missing')]:
        df[f] = df[c].isnull().astype('int8')

    df['discount'] = df['discount'].fillna(0.0).astype('float32')
    channel_is_phys = df['channel'].isin(['store', 'catalog'])
    df['tz_hint'] = np.where(df['tz_hint'].isnull(), np.where(channel_is_phys, 'local', 'UTC'), df['tz_hint'])
    df['ship_mode'] = df['ship_mode'].fillna(df['channel'].map({'store': 'in_store'})).fillna('standard')
    df['payment_method'] = df['payment_method'].fillna('card')
    df['session_clicks'] = df['session_clicks'].fillna(df['session_clicks'].median()).astype('float32')

    freight_formula = (df['weight_kg'] * 2.5 + 5.0).clip(1.0, 50.0)
    df['freight_cost'] = df['freight_cost'].fillna(freight_formula).astype('float32')

    # 3. Cyclical Temporal Harmonics (Stationary across months 1-30)
    df['hour'] = df['order_ts'].dt.hour.astype('int8')
    df['dow'] = df['order_ts'].dt.dayofweek.astype('int8')
    df['month'] = df['order_ts'].dt.month.astype('int8')
    df['date_only'] = (df['order_ts'].dt.minute == 1).astype('int8')
    df['is_weekend'] = df['dow'].isin([5, 6]).astype('int8')
    df['is_month_end'] = (df['order_ts'].dt.day >= 26).astype('int8')

    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12.0).astype(np.float32)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12.0).astype(np.float32)
    df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24.0).astype(np.float32)
    df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24.0).astype(np.float32)

    # 4. Pricing, Volume & Exposure Metrics
    df['lv'] = (df['unit_price_clean'] * df['quantity']).astype(np.float32)
    df['lq'] = np.log1p(df['quantity']).astype(np.float32)
    df['net_val'] = (df['lv'] * (1.0 - df['discount'])).astype(np.float32)
    df['price_ratio'] = (df['unit_price_clean'] / (df['list_price'] + 1e-4)).astype(np.float32)
    df['log_price_ratio'] = np.log(df['price_ratio'].clip(lower=1e-3)).astype(np.float32)
    df['discount_diff'] = (df['list_price'] - df['unit_price_clean']).clip(lower=0).astype(np.float32)
    df['is_deep_discount'] = (df['discount'] >= 0.35).astype('int8')

    df['window_x_price'] = (df['return_window_days'] * np.log1p(df['unit_price_clean'].clip(lower=0))).astype(np.float32)
    df['window_x_discount'] = (df['return_window_days'] * df['discount']).astype(np.float32)
    df['freight_share'] = (df['freight_cost'] / (df['order_value'] + 1e-4)).astype(np.float32)
    df['freight_per_unit'] = (df['freight_cost'] / (df['quantity'] + 1e-4)).astype(np.float32)
    df['freight_per_kg'] = (df['freight_cost'] / (df['weight_kg'] + 1e-4)).astype(np.float32)
    df['cross_region'] = (df['c_region'] != df['s_region']).astype('int8')

    df['has_coupon'] = df['coupon_code'].notnull().astype('int8')
    df['clicks_per_item'] = (df['session_clicks'] / (df['items_in_order'] + 1e-4)).astype(np.float32)
    df['line_pos'] = (df['line_no'] / (df['items_in_order'] + 1e-4)).astype(np.float32)

    # 5. Order-Level Multi-Item Bracketing Detection
    g = df.groupby('order_id')
    df['o_lines'] = g['order_line_id'].transform('size').astype('int16')
    df['o_qty'] = g['quantity'].transform('sum').astype('int32')
    df['o_nprod'] = g['product_id'].transform('nunique').astype('int16')
    df['o_lv'] = g['lv'].transform('sum').astype('float32')
    df['o_maxprice'] = g['unit_price_clean'].transform('max').astype(np.float32)
    df['o_minprice'] = g['unit_price_clean'].transform('min').astype(np.float32)
    df['o_price_spread'] = (df['o_maxprice'] - df['o_minprice']).astype(np.float32)

    order_cat_grp = df.groupby(['order_id', 'cat'])
    df['items_in_same_cat_in_order'] = order_cat_grp['order_line_id'].transform('count').astype(np.int16)
    df['is_bracketing_category'] = (df['items_in_same_cat_in_order'] > 1).astype('int8')

    order_supp_grp = df.groupby(['order_id', 'supplier_code'])
    df['items_from_same_supplier_in_order'] = order_supp_grp['order_line_id'].transform('count').astype(np.int16)

    df['qty_share'] = (df['quantity'] / (df['o_qty'] + 1e-4)).astype(np.float32)
    df['val_share'] = (df['lv'] / (df['o_lv'] + 1e-4)).astype(np.float32)
    df['price_rel_to_max'] = (df['unit_price_clean'] / (df['o_maxprice'] + 1e-4)).astype(np.float32)
    df['price_rank'] = g['unit_price_clean'].rank(pct=True).astype(np.float32)

    # 6. Macroeconomic Inflation & Fuel Pressures
    df['holiday_flag'] = df['holiday_flag'].fillna(0).astype('int8')
    df['temperature_f'] = df['temperature_f'].fillna(60.0).astype(np.float32)
    df['price_to_cpi'] = (df['unit_price_clean'] / (df['cpi'] + 1e-4)).astype(np.float32)
    df['cost_pressure'] = (df['fuel_price'] * (df['fuel_surcharge'].fillna(0) + 1.0)).astype(np.float32)

    # Interaction keys for target encoding
    df['promo_x_channel'] = df['promo_channel_code'].astype(str) + '_' + df['channel'].astype(str)
    df['promo_x_cat'] = df['promo_channel_code'].astype(str) + '_' + df['cat'].astype(str)

    drop_cols = [
        'signup_date', 'launch_date', 'opened_date', 'week_start',
        'record_updated_at', 'coupon_code', 'warehouse_zone', 'picker_shift'
    ]
    if 'category' in df.columns:
        drop_cols.append('category')
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    return df


# ==============================================================================
# 3. DATA INGESTION, UNSUPERVISED ENTITY STATS & FACTORIZATION
# ==============================================================================
print("\n--- [Step 3/8] Loading Data & Generating Label-Free Entity Statistics ---")

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

df_train['is_train'] = 1
df_test['is_train'] = 0
df_test['returned'] = np.nan

df_combined = pd.concat([df_train, df_test], ignore_index=True)
del df_train, df_test
gc.collect()

df_combined = build_features(df_combined)

# Label-Free Entity Aggregations (computed across train + test; zero target leakage)
for col in ['store_id', 'product_id', 'customer_id', 'supplier_code']:
    df_combined[f'{col}_n'] = df_combined.groupby(col)[col].transform('count').astype('int32')

df_combined['cust_orders'] = df_combined.groupby('customer_id')['order_id'].transform('nunique').astype('int32')
df_combined['cust_mean_qty'] = df_combined.groupby('customer_id')['quantity'].transform('mean').astype('float32')
df_combined['prod_mean_qty'] = df_combined.groupby('product_id')['quantity'].transform('mean').astype('float32')
df_combined['cust_mean_disc'] = df_combined.groupby('customer_id')['discount'].transform('mean').astype('float32')
df_combined['prod_mean_ratio'] = df_combined.groupby('product_id')['log_price_ratio'].transform('mean').astype('float32')

# Factorize categorical strings into compact integers
ignore_metadata = [
    'order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id',
    'returned', 'is_train', 'order_ts', 'promo_x_channel', 'promo_x_cat', 'supplier_code'
]

non_num_cols = [c for c in df_combined.columns if c not in ignore_metadata and not pd.api.types.is_numeric_dtype(df_combined[c])]
print(f"Factorizing {len(non_num_cols)} categorical columns to int32...")
for col in non_num_cols:
    df_combined[col] = df_combined[col].astype(str)
    cats, uniques = pd.factorize(df_combined[col])
    df_combined[col] = cats.astype(np.int32)

# Separate train and test
df_train = df_combined[df_combined['is_train'] == 1].reset_index(drop=True)
df_test = df_combined[df_combined['is_train'] == 0].reset_index(drop=True)
del df_combined
gc.collect()

# Exclude promo_channel_code from direct tree features to neutralize temporal concept drift
FEATS = [c for c in df_train.columns if c not in ignore_metadata and c != 'promo_channel_code' and pd.api.types.is_numeric_dtype(df_train[c])]
print(f"Base Features Extracted (Zero Promo Drift): {len(FEATS)}")


# ==============================================================================
# 4. EXPLORATORY DATA ANALYSIS (EDA) & CORRELATION VISUALIZATIONS
# ==============================================================================
print("\n--- [Step 4/8] Generating Exploratory Visualizations & Heatmaps ---")

eda_sample = df_train.sample(n=min(100000, len(df_train)), random_state=SEED)
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Target Distribution
sns.countplot(x='returned', data=eda_sample, ax=axes[0, 0], palette='Blues')
axes[0, 0].set_title('Target Distribution (0 = Kept, 1 = Returned)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Returned')
axes[0, 0].set_ylabel('Transaction Count')

# 2. Return Rate by Return Window Days
window_returns = df_train.groupby('return_window_days')['returned'].mean().reset_index()
sns.barplot(x='return_window_days', y='returned', data=window_returns, ax=axes[0, 1], palette='crest')
axes[0, 1].set_title('Empirical Return Rate by Return Window (Days)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Return Window (Days)')
axes[0, 1].set_ylabel('Mean Return Probability')

# 3. Cleaned Unit Price vs Return Status (Log Scale)
eda_sample['log_price'] = np.log1p(eda_sample['unit_price_clean'])
sns.boxplot(x='returned', y='log_price', data=eda_sample, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Log Cleaned Unit Price by Return Outcome', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Returned')
axes[1, 0].set_ylabel('Log(1 + Clean Price)')

# 4. Correlation Heatmap
corr_cols = [
    'returned', 'unit_price_clean', 'quantity', 'discount', 'freight_share',
    'window_price_exposure', 'is_bracketing_category', 'line_qty_share', 'order_line_count'
]
corr_matrix = eda_sample[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.3, vmax=0.3, ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlation Heatmap with Target', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


# ==============================================================================
# 5. ADAPTIVE BAYESIAN OUT-OF-FOLD TARGET ENCODING & PROMO DRIFT OFFSET
# ==============================================================================
ENTITY_SMOOTHING = {
    'customer_id': 5.0,        # Low shrinkage preserves customer habit persistence
    'product_id': 15.0,        # Medium shrinkage: product defect & sizing rate
    'store_id': 60.0,          # High shrinkage: macro channel policy
    'supplier_code': 40.0,
    'cat': 80.0,
    'promo_x_channel': 50.0,
    'promo_x_cat': 50.0
}

def apply_adaptive_oof_target_encoding(train_df, y_tr, test_df, te_configs, n_splits=5):
    """Computes Bayesian out-of-fold target encodings grouped by order_id."""
    prior = float(y_tr.mean())
    names = [f'{c}_te' for c in te_configs.keys() if c in train_df.columns]
    oof_train = {n: np.full(len(train_df), prior, dtype=np.float32) for n in names}
    test_encoded = {}

    groups = train_df['order_id'].values
    gkf = GroupKFold(n_splits=n_splits)

    for col, smoothing in te_configs.items():
        if col not in train_df.columns:
            continue
        col_te_name = f'{col}_te'

        for tr_idx, val_idx in gkf.split(train_df, y_tr, groups):
            stats = pd.DataFrame({
                'key': train_df[col].values[tr_idx],
                'target': y_tr[tr_idx]
            }).groupby('key')['target'].agg(['sum', 'count'])

            smoothed = (stats['sum'] + smoothing * prior) / (stats['count'] + smoothing)
            oof_train[col_te_name][val_idx] = (
                pd.Series(train_df[col].values[val_idx]).map(smoothed).fillna(prior).values.astype(np.float32)
            )

        full_stats = pd.DataFrame({
            'key': train_df[col].values,
            'target': y_tr
        }).groupby('key')['target'].agg(['sum', 'count'])
        full_smoothed = (full_stats['sum'] + smoothing * prior) / (full_stats['count'] + smoothing)
        test_encoded[col_te_name] = (
            pd.Series(test_df[col].values).map(full_smoothed).fillna(prior).values.astype(np.float32)
        )

    return pd.DataFrame(oof_train, index=train_df.index), pd.DataFrame(test_encoded, index=test_df.index)

def compute_promo_offset(df_part, y_part, k=50.0):
    """
    Computes smoothed month x promo logit offset (init_score).
    Allows trees to learn stable residual returns without overfitting to 2023 promo drift.
    """
    d = pd.DataFrame({
        'm': df_part['order_ts'].dt.to_period('M').astype(str).values,
        'p': df_part['promo_channel_code'].values,
        'y': y_part
    })
    d['mm'] = d.groupby('m')['y'].transform('mean')
    g = d.groupby(['m', 'p']).agg(s=('y', 'sum'), n=('y', 'size'), mm=('mm', 'first')).reset_index()
    g['r'] = (g['s'] + k * g['mm']) / (g['n'] + k)
    r = d.merge(g[['m', 'p', 'r']], on=['m', 'p'], how='left')['r'].values.clip(1e-3, 1.0 - 1e-3)
    return np.log(r / (1.0 - r)).astype('float32')


# ==============================================================================
# 6. BAYESIAN HYPERPARAMETER OPTIMIZATION (OPTUNA / BEST CONVERGED CONFIGS)
# ==============================================================================
print("\n--- [Step 5/8] Hyperparameter Tuning via Optuna / Converged Grid ---")

if OPTUNA_AVAILABLE:
    print("Running Bayesian Hyperparameter Optimization with Optuna...")
    
    # Fast temporal split for tuning: train on < 2024-07-01, validate on 2024-07-01 to 2024-10-01
    tune_tr_m = (df_train['order_ts'] < '2024-07-01').values
    tune_va_m = ((df_train['order_ts'] >= '2024-07-01') & (df_train['order_ts'] < '2024-10-01')).values
    
    # Subsample 25% of training fold for fast search
    sub_tune_idx = np.random.RandomState(SEED).choice(np.where(tune_tr_m)[0], size=int(tune_tr_m.sum() * 0.25), replace=False)
    
    df_tune_tr = df_train.iloc[sub_tune_idx].reset_index(drop=True)
    df_tune_va = df_train.loc[tune_va_m].reset_index(drop=True)
    y_tune_tr = df_tune_tr['returned'].astype(np.int32).values
    y_tune_va = df_tune_va['returned'].astype(np.int32).values
    
    te_tune_tr, te_tune_va = apply_adaptive_oof_target_encoding(df_tune_tr, y_tune_tr, df_tune_va, ENTITY_SMOOTHING)
    X_tune_tr = pd.concat([df_tune_tr[FEATS], te_tune_tr], axis=1).astype(np.float32)
    X_tune_va = pd.concat([df_tune_va[FEATS], te_tune_va], axis=1).astype(np.float32)
    off_tune_tr = compute_promo_offset(df_tune_tr, y_tune_tr)

    def lgb_objective(trial):
        p = {
            'objective': 'binary',
            'metric': 'auc',
            'boosting_type': 'gbdt',
            'learning_rate': trial.suggest_float('learning_rate', 0.025, 0.06, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 80),
            'max_depth': trial.suggest_int('max_depth', 6, 9),
            'min_child_samples': trial.suggest_int('min_child_samples', 200, 600),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.60, 0.85),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.70, 0.90),
            'bagging_freq': 1,
            'reg_alpha': trial.suggest_float('reg_alpha', 1.0, 5.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 8.0, 25.0),
            'n_jobs': N_JOBS,
            'random_state': SEED,
            'verbose': -1
        }
        d_tr = lgb.Dataset(X_tune_tr, label=y_tune_tr, init_score=off_tune_tr)
        d_va = lgb.Dataset(X_tune_va, label=y_tune_va, reference=d_tr)
        m = lgb.train(p, d_tr, 500, valid_sets=[d_va], callbacks=[lgb.early_stopping(30, verbose=False)])
        return roc_auc_score(y_tune_va, m.predict(X_tune_va))

    study = optuna.create_study(direction='maximize')
    study.optimize(lgb_objective, n_trials=10, show_progress_bar=False)
    LGB_PARAMS = study.best_params
    LGB_PARAMS.update({
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'bagging_freq': 1,
        'n_jobs': N_JOBS,
        'random_state': SEED,
        'verbose': -1
    })
    print(f"Optimal LightGBM Params Discovered: {LGB_PARAMS}")
    del df_tune_tr, df_tune_va, X_tune_tr, X_tune_va
    gc.collect()
else:
    print("Using pre-tuned, empirically validated Bayesian hyperparameters.")
    LGB_PARAMS = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.035,
        'num_leaves': 63,
        'max_depth': 8,
        'min_child_samples': 250,
        'feature_fraction': 0.70,
        'bagging_fraction': 0.80,
        'bagging_freq': 1,
        'reg_alpha': 2.0,
        'reg_lambda': 12.0,
        'n_jobs': N_JOBS,
        'random_state': SEED,
        'verbose': -1
    }

# Tuned CatBoost Hyperparameters
CB_PARAMS = {
    'iterations': 1200,
    'learning_rate': 0.04,
    'depth': 7,
    'l2_leaf_reg': 10.0,
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'early_stopping_rounds': 60,
    'verbose': 0,
    'thread_count': N_JOBS
}

# Tuned XGBoost Hyperparameters
XGB_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.04,
    'max_depth': 7,
    'min_child_weight': 30,
    'subsample': 0.80,
    'colsample_bytree': 0.70,
    'reg_alpha': 2.0,
    'reg_lambda': 12.0,
    'tree_method': 'hist',
    'n_jobs': N_JOBS,
    'random_state': SEED
}


# ==============================================================================
# 7. PURGED ROLLING TIME-BASED CROSS-VALIDATION
# ==============================================================================
print("\n--- [Step 6/8] Purged Rolling Time-Based Cross-Validation (5 Models) ---")

CV_FOLDS = [
    ('2024-04-01', '2024-07-01'),
    ('2024-07-01', '2024-10-01'),
    ('2024-10-01', '2025-01-02')
]

fold_scores = {'lgb': [], 'cb': [], 'xgb': [], 'rf': [], 'sgd': [], 'blend': []}
best_lgb_iters = []

for fold_idx, (val_start, val_end) in enumerate(CV_FOLDS, 1):
    t0 = time.time()
    tr_m = (df_train['order_ts'] < val_start).values
    va_m = ((df_train['order_ts'] >= val_start) & (df_train['order_ts'] < val_end)).values

    df_tr_fold = df_train.loc[tr_m].reset_index(drop=True)
    df_val_fold = df_train.loc[va_m].reset_index(drop=True)

    y_tr_fold = df_tr_fold['returned'].astype(np.int32).values
    y_val_fold = df_val_fold['returned'].astype(np.int32).values

    # In-fold target encoding strictly from historical transactions
    te_tr, te_va = apply_adaptive_oof_target_encoding(df_tr_fold, y_tr_fold, df_val_fold, ENTITY_SMOOTHING)
    XA = pd.concat([df_tr_fold[FEATS], te_tr], axis=1).astype(np.float32)
    XB = pd.concat([df_val_fold[FEATS], te_va], axis=1).astype(np.float32)

    # Promo logit offset
    off_A = compute_promo_offset(df_tr_fold, y_tr_fold)

    # Model 1: LightGBM (Leaf-wise GBDT)
    dA = lgb.Dataset(XA, y_tr_fold, init_score=off_A)
    dB = lgb.Dataset(XB, y_val_fold, reference=dA)
    m_lgb = lgb.train(LGB_PARAMS, dA, 2500, valid_sets=[dB], callbacks=[lgb.early_stopping(60, verbose=False)])
    p_lgb = m_lgb.predict(XB, num_iteration=m_lgb.best_iteration)
    fold_scores['lgb'].append(roc_auc_score(y_val_fold, p_lgb))
    best_lgb_iters.append(m_lgb.best_iteration)

    # Model 2: CatBoost (Oblivious Symmetric Trees)
    p_cb = np.zeros_like(p_lgb)
    if CATBOOST_AVAILABLE:
        pA = Pool(XA, y_tr_fold, baseline=off_A)
        pB = Pool(XB, y_val_fold)
        m_cb = CatBoostClassifier(**CB_PARAMS)
        m_cb.fit(pA, eval_set=pB, use_best_model=True)
        p_cb = m_cb.predict_proba(pB)[:, 1]
        fold_scores['cb'].append(roc_auc_score(y_val_fold, p_cb))

    # Model 3: XGBoost (Histogram-Based GBDT)
    p_xgb = np.zeros_like(p_lgb)
    if XGBOOST_AVAILABLE:
        dX_A = xgb.DMatrix(XA, label=y_tr_fold, base_margin=off_A)
        dX_B = xgb.DMatrix(XB, label=y_val_fold)
        m_xgb = xgb.train(XGB_PARAMS, dX_A, 1200, evals=[(dX_B, 'val')], early_stopping_rounds=60, verbose_eval=False)
        p_xgb = m_xgb.predict(dX_B, iteration_range=(0, m_xgb.best_iteration + 1))
        fold_scores['xgb'].append(roc_auc_score(y_val_fold, p_xgb))

    # Model 4: Random Forest (Bagged Variance-Reducing Trees)
    rf_idx = np.random.RandomState(SEED).choice(len(XA), size=min(250000, len(XA)), replace=False)
    m_rf = RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=50, max_features=0.5, n_jobs=N_JOBS, random_state=SEED)
    m_rf.fit(XA.iloc[rf_idx], y_tr_fold[rf_idx])
    p_rf = m_rf.predict_proba(XB)[:, 1]
    fold_scores['rf'].append(roc_auc_score(y_val_fold, p_rf))

    # Model 5: Gradient Descent (SGD / Logistic Regression on Scaled Matrix)
    scaler = StandardScaler()
    XA_sc = np.nan_to_num(scaler.fit_transform(XA), nan=0.0)
    XB_sc = np.nan_to_num(scaler.transform(XB), nan=0.0)
    m_sgd = SGDClassifier(loss='log_loss', penalty='l2', alpha=1e-4, max_iter=300, learning_rate='optimal', random_state=SEED)
    m_sgd.fit(XA_sc, y_tr_fold)
    p_sgd = m_sgd.predict_proba(XB_sc)[:, 1]
    fold_scores['sgd'].append(roc_auc_score(y_val_fold, p_sgd))

    # Percentile Rank Blend (Direct ROC-AUC Optimization)
    r_lgb = rankdata(p_lgb) / len(p_lgb)
    r_cb = rankdata(p_cb) / len(p_cb) if CATBOOST_AVAILABLE else r_lgb
    r_xgb = rankdata(p_xgb) / len(p_xgb) if XGBOOST_AVAILABLE else r_lgb
    r_rf = rankdata(p_rf) / len(p_rf)
    r_sgd = rankdata(p_sgd) / len(p_sgd)

    p_blend = 0.35 * r_lgb + 0.30 * r_cb + 0.20 * r_xgb + 0.10 * r_rf + 0.05 * r_sgd
    score_blend = roc_auc_score(y_val_fold, p_blend)
    fold_scores['blend'].append(score_blend)

    print(
        f"Fold {fold_idx} [{val_start}..{val_end}] | "
        f"LGB: {fold_scores['lgb'][-1]:.4f} | CB: {fold_scores['cb'][-1]:.4f} | "
        f"XGB: {fold_scores['xgb'][-1]:.4f} | RF: {fold_scores['rf'][-1]:.4f} | "
        f"Blend: {score_blend:.4f} ({time.time() - t0:.1f}s)"
    )

    del df_tr_fold, df_val_fold, XA, XB
    gc.collect()

print("\n" + "="*60)
print("PURGED CROSS-VALIDATION SCOREBOARD (MEAN ROC-AUC)")
print("="*60)
print(f"Mean LightGBM CV       : {np.mean(fold_scores['lgb']):.5f}")
if CATBOOST_AVAILABLE:
    print(f"Mean CatBoost CV       : {np.mean(fold_scores['cb']):.5f}")
if XGBOOST_AVAILABLE:
    print(f"Mean XGBoost CV        : {np.mean(fold_scores['xgb']):.5f}")
print(f"Mean Random Forest CV  : {np.mean(fold_scores['rf']):.5f}")
print(f"Mean GradientDescent CV: {np.mean(fold_scores['sgd']):.5f}")
print(f"Mean Quintuple Blend   : {np.mean(fold_scores['blend']):.5f}")
print("="*60)


# ==============================================================================
# 8. FULL RETRAINING ACROSS MONTHS 1-24 & TEST PREDICTIONS
# ==============================================================================
print("\n--- [Step 7/8] Retraining Models on Full Historical Data (Months 1-24) ---")

y_full = df_train['returned'].astype(np.int32).values

# Target encoding computed across all historical orders
te_full, te_test = apply_adaptive_oof_target_encoding(df_train, y_full, df_test, ENTITY_SMOOTHING)
X_full = pd.concat([df_train[FEATS], te_full], axis=1).astype(np.float32)
X_test = pd.concat([df_test[FEATS], te_test], axis=1).astype(np.float32)

off_full = compute_promo_offset(df_train, y_full)

# 1. Full LightGBM with 3-seed variance reduction
optimal_lgb_rounds = int(np.mean(best_lgb_iters) * 1.15)
lgb_test_preds = np.zeros(len(df_test), dtype=np.float64)
print(f"Retraining LightGBM (3 seeds, {optimal_lgb_rounds} rounds)...")
for s in [SEED, SEED + 1, SEED + 2]:
    p = dict(LGB_PARAMS, random_state=s)
    ds_full = lgb.Dataset(X_full, label=y_full, init_score=off_full)
    mdl = lgb.train(p, ds_full, num_boost_round=optimal_lgb_rounds)
    lgb_test_preds += (rankdata(mdl.predict(X_test)) / len(X_test)) / 3.0

# 2. Full CatBoost
cb_test_preds = np.zeros(len(df_test), dtype=np.float64)
if CATBOOST_AVAILABLE:
    print("Retraining CatBoost on full historical data...")
    pool_full = Pool(X_full, y_full, baseline=off_full)
    pool_test = Pool(X_test)
    cbp = dict(CB_PARAMS)
    cbp.pop('early_stopping_rounds', None)
    cbp['iterations'] = 1000
    m_cb_full = CatBoostClassifier(**cbp)
    m_cb_full.fit(pool_full)
    cb_test_preds = rankdata(m_cb_full.predict_proba(pool_test)[:, 1]) / len(X_test)

# 3. Full XGBoost
xgb_test_preds = np.zeros(len(df_test), dtype=np.float64)
if XGBOOST_AVAILABLE:
    print("Retraining XGBoost on full historical data...")
    dX_full = xgb.DMatrix(X_full, label=y_full, base_margin=off_full)
    dX_test = xgb.DMatrix(X_test)
    m_xgb_full = xgb.train(XGB_PARAMS, dX_full, num_boost_round=1000)
    xgb_test_preds = rankdata(m_xgb_full.predict(dX_test)) / len(X_test)

# 4. Full Random Forest
print("Retraining Random Forest on full historical data...")
m_rf_full = RandomForestClassifier(n_estimators=200, max_depth=14, min_samples_leaf=40, max_features=0.5, n_jobs=N_JOBS, random_state=SEED)
rf_full_idx = np.random.RandomState(SEED).choice(len(X_full), size=min(350000, len(X_full)), replace=False)
m_rf_full.fit(X_full.iloc[rf_full_idx], y_full[rf_full_idx])
rf_test_preds = rankdata(m_rf_full.predict_proba(X_test)[:, 1]) / len(X_test)

# 5. Full Gradient Descent (SGD)
print("Retraining Gradient Descent (SGD) on scaled matrix...")
scaler_full = StandardScaler()
X_full_sc = np.nan_to_num(scaler_full.fit_transform(X_full), nan=0.0)
X_test_sc = np.nan_to_num(scaler_full.transform(X_test), nan=0.0)

m_sgd_full = SGDClassifier(loss='log_loss', penalty='l2', alpha=1e-4, max_iter=350, learning_rate='optimal', random_state=SEED)
m_sgd_full.fit(X_full_sc, y_full)
sgd_test_preds = rankdata(m_sgd_full.predict_proba(X_test_sc)[:, 1]) / len(X_test)

# Final Percentile Rank Ensemble Blend
if CATBOOST_AVAILABLE and XGBOOST_AVAILABLE:
    final_predictions = 0.35 * lgb_test_preds + 0.30 * cb_test_preds + 0.20 * xgb_test_preds + 0.10 * rf_test_preds + 0.05 * sgd_test_preds
elif CATBOOST_AVAILABLE:
    final_predictions = 0.45 * lgb_test_preds + 0.35 * cb_test_preds + 0.15 * rf_test_preds + 0.05 * sgd_test_preds
else:
    final_predictions = 0.70 * lgb_test_preds + 0.20 * rf_test_preds + 0.10 * sgd_test_preds


# ==============================================================================
# 9. SUBMISSION GENERATION & VERIFICATION
# ==============================================================================
print("\n--- [Step 8/8] Generating & Validating submission.csv ---")

submission = pd.DataFrame({
    'order_line_id': df_test['order_line_id'],
    'returned': final_predictions
})

submission.to_csv('submission4.csv', index=False)
print("Saved submission4.csv successfully!")
print(f"Total Rows       : {len(submission):,}")
print(f"Missing Values   : {submission['returned'].isnull().sum()}")
print(f"Probability Range: [{submission['returned'].min():.5f}, {submission['returned'].max():.5f}]")
print("\nFirst 10 Predictions:")
print(submission.head(10))

c:\python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Directory    : .
LightGBM Version  : 4.7.0
CatBoost Ready    : True
XGBoost Ready     : True
Optuna Ready      : True

--- [Step 1/8] Cleaning & Logically Imputing Supplementary Tables ---


NameError: name 'missing_weight' is not defined